# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 02: Training — Experiment 3 (Joint Training)

**Author:** Binger Yu (Data & Preprocessing Lead + Experiment 3 Lead)   
**Datasets:** COD10K + CAMO + ACD1K   
**Model:** SegFormer-B2 (nvidia/segformer-b2-finetuned-ade-512-512)   
**Purpose:** Train SegFormer-B2 jointly on all three datasets in a single
training run and evaluate whether enriching the training distribution with both animal and artificial camouflage data yields gains over sequential transfer learning (Experiment 2). This notebook covers hyperparameter sweeping, final training, and result logging.

---

### Notebook Structure

1. **Environment Setup** — Clone repo, install dependencies, configure Kaggle credentials
2. **Dataset Download** — COD10K (Kaggle), ACD1K (Kaggle), CAMO (Google)
3. **Dataset Verification** — Confirm all folder paths and split index files
4. **GPU Verification** — Confirm A100 is available
5. **Hyperparameter Sweep** — Three 20-epoch runs across learning rates [1e-4, 6e-5, 1e-5]
6. **Sweep Results** — Compare val mIoU across runs, select best LR
7. **Final Training Run** — 50-epoch run with best LR, early stopping patience=10
8. **Results & Export** — Save history.json, push to GitHub, download checkpoint

---

### Experiment Design

| Setting | Value |
|---|---|
| Model | SegFormer-B2 |
| Pretrained checkpoint | nvidia/segformer-b2-finetuned-ade-512-512 |
| Training data | COD10K (6,000) + CAMO (1,000) + ACD1K (748) = 7,748 images |
| Validation data | COD10K val (3,950) + CAMO val (250) + ACD1K val (230) = 4,430 images |
| Input resolution | 512 × 512 |
| Batch size | 16 (A100) |
| ACD1K oversample weight | 8.0 (WeightedRandomSampler) |
| Loss function | Binary cross-entropy + structural similarity |
| Sweep LRs | 1e-4, 6e-5, 1e-5 |
| Final LR | 6e-5 (best from final) |
| Early stopping patience | 10 epochs |
| Hardware | Google Colab A100 GPU |

## 1. Environment Setup

In [1]:
# Cell 1 — Clone repo
!git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

Cloning into 'AI-final-project'...
remote: Enumerating objects: 274, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 274 (delta 29), reused 39 (delta 20), pack-reused 220 (from 2)
Receiving objects: 100% (274/274), 48.01 MiB | 11.53 MiB/s, done.
Resolving deltas: 100% (116/116), done.
/content/AI-final-project


In [2]:
# Cell 2 — Install dependencies
!pip install -q transformers albumentations kaggle

## 2. Dataset Download

In [3]:
# Cell 3 — Upload kaggle.json (run once per session)
import os
from google.colab import files

files.upload()  # select kaggle.json from your MacBook when prompted
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured')

Saving kaggle.json to kaggle.json
Kaggle credentials configured


In [4]:
# Cell 4 — Download COD10K (~1.2 GB)
!kaggle datasets download \
    -d ivanomelchenkoim11/cod10k-dataset \
    -p data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/ivanomelchenkoim11/cod10k-dataset
License(s): unknown
100% 2.26G/2.26G [02:24<00:00, 16.7MB/s]



In [5]:
# Cell 5 — Download ACD1K (~350 MB)
!kaggle datasets download \
    -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
    -p data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/aalihhiader/military-camouflage-soldiers-dataset-mcs1k
License(s): CC0-1.0
100% 372M/372M [00:28<00:00, 13.9MB/s]



In [6]:
# Cell 6 - Download CAMO from GitHub releases
!wget -q --show-progress \
    https://github.com/bing-er/AI-final-project/releases/download/v1.0/CAMO-V.1.0-CVIU2019.zip \
    -O data/CAMO-V.1.0-CVIU2019.zip

# Force overwrite, skip Mac junk files
!unzip -q -o data/CAMO-V.1.0-CVIU2019.zip \
    -d data/ \
    -x "*/__MACOSX/*" \
    -x "*/.DS_Store" \
    -x "*/._.DS_Store"

!rm data/CAMO-V.1.0-CVIU2019.zip
!rm -rf data/__MACOSX  # remove if it snuck in

# Verify
import os
for p in ['data/CAMO-V.1.0-CVIU2019/Images/Train',
          'data/CAMO-V.1.0-CVIU2019/Images/Test',
          'data/CAMO-V.1.0-CVIU2019/GT']:
    print('✅' if os.path.exists(p) else '❌', p)

data/CAMO-V.1.0-CVI 100%[===================>] 204.16M  35.3MB/s    in 5.8s    
caution: excluded filename not matched:  */__MACOSX/*
caution: excluded filename not matched:  -x
caution: excluded filename not matched:  -x
✅ data/CAMO-V.1.0-CVIU2019/Images/Train
✅ data/CAMO-V.1.0-CVIU2019/Images/Test
✅ data/CAMO-V.1.0-CVIU2019/GT


## 3. Dataset Verification

In [7]:
# Cell 7 — Verify dataset loading
!PYTHONPATH=/content/AI-final-project/src \
 python /content/AI-final-project/src/dataset.py \
    /content/AI-final-project/data/ \
    /content/AI-final-project/splits/

Verifying all conditions and splits...

--- ACD1K / train ---
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=187
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'ACD1K'}
  ✅ OK

--- ACD1K / val ---
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=58
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'ACD1K'}
  ✅ OK

--- COD10K / train ---
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [DataLoader] condition=cod10k split=train samples=5950 batches=1487
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'COD10K'}
  ✅ OK

--- COD10K / val ---
  [Splits] COD10K val: 3950 images (from cod10k_split

In [8]:
# Cell 8 — Verify folder structure
import os
expected = [
    'data/COD10K-v3/Train/Image',
    'data/COD10K-v3/Train/GT_Object',
    'data/COD10K-v3/Test/Image',
    'data/COD10K-v3/Test/GT_Object',
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
    'data/CAMO-V.1.0-CVIU2019/Images/Train',
    'data/CAMO-V.1.0-CVIU2019/Images/Test',
    'data/CAMO-V.1.0-CVIU2019/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

✅ — data/COD10K-v3/Train/Image
✅ — data/COD10K-v3/Train/GT_Object
✅ — data/COD10K-v3/Test/Image
✅ — data/COD10K-v3/Test/GT_Object
✅ — data/dataset-splitM/Training/images
✅ — data/dataset-splitM/Training/GT
✅ — data/dataset-splitM/Testing/images
✅ — data/dataset-splitM/Testing/GT
✅ — data/CAMO-V.1.0-CVIU2019/Images/Train
✅ — data/CAMO-V.1.0-CVIU2019/Images/Test
✅ — data/CAMO-V.1.0-CVIU2019/GT


## 4. GPU Verification

In [9]:
# Cell 9 — Verify GPU
!nvidia-smi

Sun Apr  5 13:34:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 5. Hyperparameter Sweep

In [10]:
# Cell 10 — Sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 1e-4 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr1e4

Device: cuda
Config saved → outputs/exp3/sweep_lr1e4/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/sweep_lr1e4",
  "lr": 0.0001,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [CAMO] 1000 images (folder-scan mode)
  [ACD1K] 748 images (folder-scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=481
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
config.json: 6.88kB [00:00, 16.5MB/s]
pytorch_model.bin: 100% 110M/110M [00:02<00:00, 41.9MB/s]
Loading weights: 100% 380/380 [00:00<00:00, 746.90i

In [11]:
# Cell 11 - Show sweep run 1 results
import json

with open('outputs/exp3/sweep_lr1e4/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Run 1 (lr=1e-4) on A100:")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Run 1 (lr=1e-4) on A100:
  Best epoch : 6
  val mIoU   : 0.8683
  val F1     : 0.8613
  val MAE    : 0.0383


In [12]:
# Cell 12 — Sweep run 2 (lr=6e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 6e-5 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr6e5

Device: cuda
Config saved → outputs/exp3/sweep_lr6e5/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/sweep_lr6e5",
  "lr": 6e-05,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [CAMO] 1000 images (folder-scan mode)
  [ACD1K] 748 images (folder-scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=481
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 831.71it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSe

In [13]:
# Cell 13 - Show sweep run 2 results
import json

with open('outputs/exp3/sweep_lr6e5/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Run 1 (lr=6e-05) on A100:")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Run 1 (lr=6e-05) on A100:
  Best epoch : 7
  val mIoU   : 0.8736
  val F1     : 0.8691
  val MAE    : 0.0362


In [14]:
# Cell 14 — Sweep run 3 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 1e-5 --acd1k_w 8.0 --epochs 20 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/sweep_lr1e5

Device: cuda
Config saved → outputs/exp3/sweep_lr1e5/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/sweep_lr1e5",
  "lr": 1e-05,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [CAMO] 1000 images (folder-scan mode)
  [ACD1K] 748 images (folder-scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=481
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 848.93it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSe

In [15]:
# Cell 15 - Show sweep run 3 results
import json

with open('outputs/exp3/sweep_lr1e5/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Run 1 (lr=1e-05) on A100:")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Run 1 (lr=1e-05) on A100:
  Best epoch : 18
  val mIoU   : 0.8647
  val F1     : 0.8574
  val MAE    : 0.0417


## 6. Sweep Results

In [16]:
# Cell 16 — Check sweep results, pick best LR
import json
for run in ['sweep_lr1e4', 'sweep_lr6e5', 'sweep_lr1e5']:
    path = f'outputs/exp3/{run}/history.json'
    if not os.path.exists(path):
        print(f'{run}: not found')
        continue
    with open(path) as f:
        h = json.load(f)
    best = max(h, key=lambda x: x['val_mIoU'])
    print(f"{run}: best val mIoU={best['val_mIoU']:.4f} @ epoch {best['epoch']}")

sweep_lr1e4: best val mIoU=0.8683 @ epoch 6
sweep_lr6e5: best val mIoU=0.8736 @ epoch 7
sweep_lr1e5: best val mIoU=0.8647 @ epoch 18


## 7. Final Training Run

In [17]:
# Cell 17 — Final training run (A100)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp3.py \
    --lr 6e-05 --acd1k_w 8.0 --epochs 50 \
    --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp3/final


Device: cuda
Config saved → outputs/exp3/final/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp3/final",
  "lr": 6e-05,
  "acd1k_w": 8.0,
  "weight_decay": 0.0001,
  "epochs": 50,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [CAMO] 1000 images (folder-scan mode)
  [ACD1K] 748 images (folder-scan mode)
  [Sampler] ACD1K weight=8.0, others=1.0
  [DataLoader] condition=joint split=train samples=7698 batches=481
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 788.28it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation L

## 8. Results & Export

In [18]:
# Cell 18 - Push all results to GitHub
!git config --global user.email "gyu42@my.bcit.ca"
!git config --global user.name "bing-er"
!git add outputs/exp3/sweep_lr1e4/history.json \
         outputs/exp3/sweep_lr6e5/history.json \
         outputs/exp3/sweep_lr1e5/history.json \
         outputs/exp3/final/history.json \
         outputs/exp3/final/config.json
!git commit -m "Exp3 all sweep results + final training (A100, lr=6e-5, batch=16)"
!git push

# Download checkpoint and results
from google.colab import files
files.download('outputs/exp3/final/best_model.pth')
files.download('outputs/exp3/final/history.json')

[main d99a505] Exp3 all sweep results + final training (A100, lr=6e-5, batch=16)
 4 files changed, 836 insertions(+), 920 deletions(-)
 rewrite outputs/exp3/final/history.json (78%)
 rewrite outputs/exp3/sweep_lr1e4/history.json (84%)
 rewrite outputs/exp3/sweep_lr1e5/history.json (76%)
 rewrite outputs/exp3/sweep_lr6e5/history.json (83%)
fatal: could not read Username for 'https://github.com': No such device or address


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# Cell 19 - Show experiment 3 full run results
import json, os

runs = {
    'sweep_lr1e4':  'outputs/exp3/sweep_lr1e4/history.json',
    'sweep_lr6e5':  'outputs/exp3/sweep_lr6e5/history.json',
    'sweep_lr1e5':  'outputs/exp3/sweep_lr1e5/history.json',
    'final_lr6e5':  'outputs/exp3/final/history.json',
}

print("=== Experiment 3 Full Results ===")
for name, path in runs.items():
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        best = max(h, key=lambda x: x['val_mIoU'])
        print(f"{name:<20} mIoU={best['val_mIoU']:.4f}  "
              f"F1={best['val_F1']:.4f}  "
              f"MAE={best['val_MAE']:.4f}  "
              f"@ ep{best['epoch']}")
    else:
        print(f"{name:<20} NOT FOUND")

=== Experiment 3 Full Results ===
sweep_lr1e4          mIoU=0.8683  F1=0.8613  MAE=0.0383  @ ep6
sweep_lr6e5          mIoU=0.8736  F1=0.8691  MAE=0.0362  @ ep7
sweep_lr1e5          mIoU=0.8647  F1=0.8574  MAE=0.0417  @ ep18
final_lr6e5          mIoU=0.8751  F1=0.8704  MAE=0.0387  @ ep9


In [21]:
# Cell 20 - Check history of final run
import json
with open('outputs/exp3/final/history.json') as f:
    h = json.load(f)
best = max(h, key=lambda x: x['val_mIoU'])
print(f"Best epoch : {best['epoch']}")
print(f"val mIoU   : {best['val_mIoU']:.4f}")
print(f"val F1     : {best['val_F1']:.4f}")
print(f"val MAE    : {best['val_MAE']:.4f}")
print(f"val loss   : {best['val_loss']:.4f}")

Best epoch : 9
val mIoU   : 0.8751
val F1     : 0.8704
val MAE    : 0.0387
val loss   : 0.2307
